In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
from music2latent.hparams import hparams


hparams.conv_mode = "causal"

In [ ]:

from music2latent.models_stream import *
net = UNet()

dummy = 
scripted = torch.jit.script(net)

In [4]:
# basic imports
import numpy as np
import IPython
import librosa
import os 
os.environ["CUDA_VISIBLE_DEVICES"]="-1"

from music2latent import EncoderDecoder

encdec = EncoderDecoder(load_path_inference = "/data/nils/repos/codecs_benchmark/music2latent/checkpoints/2025-05-22 17:50:44.950460/model_fid_1.0834037859147487_loss_66.655_iters_50000.pt")

In [ ]:

wv=  np.linspace(0, 1, 131072)
print(f'waveform samples: {wv.shape}')

latent = encdec.encode(wv)
print(f'Shape of latents: {latent.shape}')

# Inference

In [10]:
from torchaudio.transforms import Spectrogram, InverseSpectrogram
import torch
class StreamableSTFT(torch.nn.Module):
    
    def __init__(self,
                 nfft=1024,
                 hop_size=256,
                 stream=False,
                 skip_features=None,
                 log1p=False,
                 alpha_rescale=0.65,
                 beta_rescale=0.34):
        super().__init__()
        self.nfft = nfft
        self.hop_size = hop_size
        self.stream = stream
        self.log1p = log1p
        self.skip_features = skip_features
        self.alpha_rescale = alpha_rescale
        self.beta_rescale = beta_rescale
        self.nskip = 1
        self.register_buffer('audio_buffer',
                             torch.zeros((1, 1, nfft - hop_size)))

        self.transform = Spectrogram(n_fft=nfft,
                                     win_length=nfft,
                                     hop_length=hop_size,
                                     center=not stream,
                                     normalized=False,
                                     power=None)

        self.inverse_transform = ISTFT(n_fft=nfft,
                                       win_length=nfft,
                                       hop_length=hop_size,
                                       padding="same" if stream else "center")

    def normalize_complex(self, x):
        return (self.beta_rescale *
                (x.abs()**self.alpha_rescale).to(torch.complex64) *
                torch.exp(1j * torch.angle(x).to(torch.complex64)))

    def denormalize_complex(self, x):
        x = x / self.beta_rescale
        return (x.abs()**(1.0 / self.alpha_rescale)).to(
            torch.complex64) * torch.exp(
                1j * torch.angle(x).to(torch.complex64))

    @torch.jit.export
    def forward(self, x):
        # X : B x hop_size
        if self.stream == True:
            if self.audio_buffer.shape[0] != x.shape[0]:
                print(
                    "Resizing and resetting buffer - the batch size has changed"
                )
                self.audio_buffer = torch.zeros(
                    (x.shape[0], 1, self.nfft - self.hop_size)).to(x)

            x = torch.cat([self.audio_buffer, x], dim=-1)
            self.audio_buffer = x[..., -(self.nfft - self.hop_size):]

        spec = self.transform(x)

        spec = spec if self.stream else spec[..., :-1]
        spec = self.normalize_complex(spec)

        if self.skip_features is not None:
            spec = spec[:, :, self.skip_features:]  #Drop constant componnet

        return torch.cat((torch.real(spec), torch.imag(spec)), -3)

    def inverse(self, spec):

        real, imag = torch.chunk(spec, 2, -3)

        spec = torch.complex(real.squeeze(-3), imag.squeeze(-3))
        spec = self.denormalize_complex(spec)

        spec = spec.unsqueeze(1)

        if self.skip_features is not None:
            spec = torch.cat(
                (torch.zeros_like(spec)[:, :, :self.skip_features], spec), -2)

        spec = spec.squeeze(1)

        if self.stream == False:
            spec = torch.cat((spec, torch.zeros_like(spec)[:, :, :1]), -1)

        x = self.inverse_transform(spec.squeeze(1)).unsqueeze(1)
        return x
    
    

class ISTFT(torch.nn.Module):
    """
    Custom implementation of ISTFT since torch.istft doesn't allow custom padding (other than `center=True`) with
    windowing. This is because the NOLA (Nonzero Overlap Add) check fails at the edges.
    See issue: https://github.com/pytorch/pytorch/issues/62323
    Specifically, in the context of neural vocoding we are interested in "same" padding analogous to CNNs.
    The NOLA constraint is met as we trim padded samples anyway.

    Args:
        n_fft (int): Size of Fourier transform.
        hop_length (int): The distance between neighboring sliding window frames.
        win_length (int): The size of window frame and STFT filter.
        padding (str, optional): Type of padding. Options are "center" or "same". Defaults to "same".
    """

    def __init__(self,
                 n_fft: int,
                 hop_length: int,
                 win_length: int,
                 padding: str = "same"):
        super().__init__()
        if padding not in ["center", "same"]:
            raise ValueError("Padding must be 'center' or 'same'.")
        self.padding = padding
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.win_length = win_length
        window = torch.hann_window(win_length)
        self.register_buffer("window", window)

    def forward(self, spec: torch.Tensor) -> torch.Tensor:
        """
        Compute the Inverse Short Time Fourier Transform (ISTFT) of a complex spectrogram.

        Args:
            spec (Tensor): Input complex spectrogram of shape (B, N, T), where B is the batch size,
                            N is the number of frequency bins, and T is the number of time frames.

        Returns:
            Tensor: Reconstructed time-domain signal of shape (B, L), where L is the length of the output signal.
        """
        if self.padding == "center":
            # Fallback to pytorch native implementation
            return torch.istft(spec,
                               self.n_fft,
                               self.hop_length,
                               self.win_length,
                               self.window,
                               center=True)
        elif self.padding == "same":
            pad = (self.win_length - self.hop_length) // 2
        else:
            raise ValueError("Padding must be 'center' or 'same'.")

        assert spec.dim() == 3, "Expected a 3D tensor as input"
        B, N, T = spec.shape

        # Inverse FFT
        ifft = torch.fft.irfft(spec, self.n_fft, dim=1, norm="backward")
        ifft = ifft * self.window[None, :, None]

        # Overlap and Add
        output_size = (T - 1) * self.hop_length + self.win_length
        y = torch.nn.functional.fold(
            ifft,
            output_size=(1, output_size),
            kernel_size=(1, self.win_length),
            stride=(1, self.hop_length),
        )[:, 0, 0, pad:-pad]

        # Window envelope
        window_sq = self.window.square().expand(1, T, -1).transpose(1, 2)
        window_envelope = torch.nn.functional.fold(
            window_sq,
            output_size=(1, output_size),
            kernel_size=(1, self.win_length),
            stride=(1, self.hop_length),
        ).squeeze()[pad:-pad]

        # Normalize
        assert (window_envelope > 1e-11).all()
        y = y / window_envelope

        return y



In [ ]:

import torch.nn as nn 

class EncoderDecoderStream(nn.Module):
    def __init__(self, net):
        
        super().__init__()
        self.net = net 
        
        self.hop = 256
        self.fac = 4
        self.stft = StreamableSTFT()
        
    def encode(self, x):
        S = self.stft(x)    
        return S 
        

        
Net  = EncoderDecoderStream(net = scripted)
audio = torch.randn(1, 1, 131072)

Net.encode(audio).shape

## Cached Conv 2D

In [1]:
%load_ext autoreload
%autoreload 2


In [4]:

def chunk_process(f, x, N):
    x = torch.split(x, x.shape[-1] // N, -1)
    y = torch.cat([f(_x) for _x in x], -1)
    return y


def test_equal(model_constructor, input_tensor, crop=0):
    
    cc.use_cached_conv(False)
    
    print(cc.USE_BUFFER_CONV)
    model = model_constructor()
    cc.use_cached_conv(True)
    cmodel = model_constructor()

    for p1, p2 in zip(model.parameters(), cmodel.parameters()):
        p2.data.copy_(p1.data)
    y = model(input_tensor)
    if cmodel.cumulative_delay>0:
        y = y[..., :-cmodel.cumulative_delay]
        
        
    _ = cmodel(input_tensor)
    
    cy = chunk_process(lambda x: cmodel(x), input_tensor,
                       4)[..., cmodel.cumulative_delay:]
    
    # cy = chunk_process(lambda x: cmodel(x), input_tensor,
    #                    4)[..., cmodel.cumulative_delay:]

    if type(crop)==bool:
        cd = cmodel.cumulative_delay
        y = y[..., cd:-cd]
        cy = cy[..., cd:-cd]
    else:
        y = y[...,crop:]
        cy = cy[...,crop:]

    
    
    print(y.shape, cy.shape)
    return torch.allclose(y, cy, 1e-4, 1e-4)

In [ ]:
from music2latent.cached_conv_2d import CachedConv2d, CausalConv2d, CachedGroupNorm
import cached_conv as cc
import torch

kernel_size = 3
stride = 2

# conv = CausalConv2d(1, 1, kernel_size = kernel_size, stride = stride, padding_time = cc.get_padding(kernel_size = kernel_size), padding_vert = (kernel_size-1)//2)
cc.use_cached_conv(True)



spec= torch.randn(1, 1 ,16,16)


conv = CachedConv2d(1, 1, kernel_size =kernel_size, stride =stride, padding_time = cc.get_padding(kernel_size = kernel_size, stride= stride, mode= "causal"), padding_vert = "same")


spec_out = conv(spec)

print(spec_out.shape)
print(spec_out)

In [ ]:

kernel_size = 5
stride = 2


model_constructor = lambda : CachedConv2d(1, 1, kernel_size =kernel_size, stride =stride, padding_time = cc.get_padding(kernel_size = kernel_size, stride = stride, mode = "causal"), padding_vert = (kernel_size-1)//2)

spec= torch.randn(1, 1 ,128,128)
test_equal(model_constructor, input_tensor=spec, crop=2)

In [ ]:


x = torch.randn(1, 1, 1, 10)

cc.use_cached_conv(True)
layer = CachedGroupNorm(padding = "automatic",num_groups = 1, num_channels = 1)
xn = layer(x[...,:10])
xn = layer(x[...,:8])

xn = layer(x[...,8:10])
print(xn)

cc.use_cached_conv(False)
layer = CachedGroupNorm(padding = "automatic",num_groups = 1, num_channels = 1)
xn = layer(x)
print(xn)

In [5]:
from music2latent.models_stream import *
hparams.conv_mode=  "causal"
net = UNet()
# scripted = t#orch.jit.script(net)

In [6]:
from music2latent.hparams import hparams

from music2latent.models_stream import *

hparams.pre_normalize_2d_to_1d = True
hparams.normalization = True
hparams.pre_normalize_downsampling_encoder = True
hparams.conv_mode = "causal"

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = UNet().eval()
        self.cumulative_delay = 0
    def forward(self, x):
        latents = self.net.encoder(x)
        # return latents
        pyramid_latents = self.net.decoder(latents)
        
        noisy_samples = x
        sigmas_step = None
        fdata =  self.net.forward_generator(latents, noisy_samples, sigmas_step, pyramid_latents).detach()
        return fdata
        
    
model_constructor = lambda : Encoder()

spec = torch.randn(1, 2, 1024, 128)
test_equal(model_constructor, input_tensor=spec, crop=64)

False
torch.Size([1, 2, 1024, 64]) torch.Size([1, 2, 1024, 64])


True

In [7]:
from music2latent.hparams import hparams

from music2latent.models_stream import *

hparams.pre_normalize_2d_to_1d = True
hparams.normalization = True
hparams.pre_normalize_downsampling_encoder = True
hparams.conv_mode = "causal"


# hparams.pre_normalize_2d_to_1d = False
# hparams.normalization = False
# hparams.pre_normalize_downsampling_encoder = False
# hparams.conv_mode = "causal"

net = Encoder()
spec = torch.randn(1, 2, 1024, 128)
net(spec)
scripted = torch.jit.script(net)

In [8]:
allel = 0
for p in scripted.parameters():
    allel+=p.numel()
print(allel/1e6)

16.261568


In [9]:
torch.jit.save(scripted, "scripted_m2l_big.ts")

In [ ]:
import time
st = time.time()

scripted(spec)

end = time.time()-st
print(end)

print(spec.shape[-1]*512/44100)

In [ ]:
net = Encoder()

In [14]:
x = 64



64

In [9]:
from after.dataset import SimpleDataset
# , "/data/nils/datasets/electronic/techno_dataset_v2/audio_44k" ,"/data/nils/datasets/electronic/tipper"

ds = ["/data/nils/datasets/drums/export/expended_gmd/audio_44k", "/data/nils/datasets/drums/export/darbouka/audio_44k","/data/nils/datasets/drums/export/breaks/audio_44k","/data/nils/datasets/electronic/canblast/audio_44k", "/data/nils/datasets/music_dataset_copy/lofi/", "/data/nils/datasets/music_dataset_copy/rock/","/data/nils/datasets/music_dataset_copy/dub/", "/data/nils/datasets/music_dataset_copy/jazz/","/data/nils/datasets/raw/audio_44k","/data/nils/datasets/instruments/export/maestro-v3.0.0/", "/data/nils/datasets/instruments/export/slakh/", "/data/nils/datasets/instruments/export/violin/", "/data/nils/datasets/instruments/export/guitarset"]
ds = ["/data/nils/datasets/electronic/techno_dataset_v2/audio_44k"]
for p in ds:
    dataset = SimpleDataset(p, keys=["waveform"])
    print(dataset.get_keys())
    print(p, dataset[0])
    

['waveform', 'metadata']
/data/nils/datasets/electronic/techno_dataset_v2/audio_44k {'waveform': array([ 0.        ,  0.        ,  0.        , ..., -0.02386547,
       -0.04214606, -0.0245674 ], dtype=float32)}


In [ ]:

cc.use_cached_conv(False)
net = Encoder()


In [ ]:
net(spec[...,:14])